# 7000LBSBAI - Programming for Business Analytics
## Week 11 - Write-along: From Notebook to Report

---

**Module:** Programming for Business Analytics  
**Week:** 11  
**Time:** About 60 minutes (four blocks, with your tutor)  
**Topics covered:** Turning outputs into findings · Explaining model results in plain English · Recommendations and limitations · The executive summary

---

### How to use this notebook

**This week you do not write any code.** All the code is written for you: run each cell, read the output, and write about it.

| Symbol | Meaning |
|--------|---------|
| ✍️ Run this cell | Code is already written -> just run it |
| 👀 Watch | Your tutor shows how to write about an output first |
| ✏️ Draft | Write your own version in the empty cell |
| 🔧 Improve | Check your draft against the checklist, then edit it |
| 🔴 Extension | For students who finish early (optional) |
| 📝 Hint/Tip | Some friendly help and advice |

To run a cell: click on it and press **Shift + Enter**. To write in a text cell: **double-click** it.

---

## ⚠️ BEFORE YOU CONTINUE

### Save a copy of this notebook

Make sure your notebook is saved to your Google Drive:
- **File → Save a copy in Drive**
- Name it: `Week11_WriteAlong_Notebook_to_Report_YourName.ipynb`

---
## How this write-along works

Four blocks, each about 15 minutes: **👀 Watch → ✏️ Draft → 🔧 Improve → ✅ Check**.

1. **👀 Watch:** your tutor turns one output into report writing, and explains each choice.
2. **✏️ Draft:** you write your own version for a different output.
3. **🔧 Improve:** you check your draft against a short checklist and edit it.
4. **✅ Check:** model answers are shared after Block 4. Compare them with yours.

- Finished early? Try the **🔴 Go further** challenge.

---
## The scenario

You have finished your analysis of **FitLife Gyms** (Weeks 8 to 10). The membership manager does not read Python. She wants a short report:

> ### *"Who is cancelling, when, and what should we do about it?"*

| Block | What you write |
|---|---|
| 1 | A **finding**: headline, evidence, so what |
| 2 | **Model results** in plain English |
| 3 | A **recommendation** and a **limitation** |
| 4 | The **executive summary** |

> 📝 Everything you practise here is what you will do with your **NovaTel** notebook in Assessment 2.

---
## Set up: run this cell

This cell loads and cleans FitLife (the Week 8 function), creates `cancelled_flag` and `tenure_months` (Week 9), and fits the cancellation model (Week 10).

In [ ]:
# ✍️ Set up: just run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Set the seaborn chart style
sns.set_theme(style='whitegrid')

# Import the FitLife Members data
raw = pd.read_csv("https://raw.githubusercontent.com/7000lbsbai/7000LBSBAI/main/datasets/fitlife_members.csv")

## PLEASE NOTE: The following 4 steps should be included in separate code cells
## later in your own EDA analysis for Assessment 2 at the appropriate stage.
## They are included here for convenience for today's task only.

# 1. Create a function to clean the raw FitLife member data
def clean_members(data):
    """Clean the FitLife member data (the Week 8 function)."""
    data = data.copy()
    data = data.drop_duplicates()
    data['membership_type'] = data['membership_type'].str.strip().str.title()
    data['monthly_fee'] = pd.to_numeric(data['monthly_fee'].str.replace('£', ''))
    data = data[(data['age'] >= 18) & (data['age'] <= 100)]
    data['visits_per_month'] = data['visits_per_month'].fillna(data['visits_per_month'].median())
    return data

# Run the clean_members function on the raw dataset
df = clean_members(raw)

# 2. The outcome (Week 9): 1 if the member has a cancel date
df['cancelled_flag'] = np.where(df['cancel_date'].notna(), 1, 0)

# 3. How long each member has been with FitLife (Week 8), up to the end of the data
df['join_date'] = pd.to_datetime(df['join_date'])
df['cancel_date'] = pd.to_datetime(df['cancel_date'])
end_date = df['cancel_date'].fillna(pd.Timestamp('2025-01-31'))
df['tenure_months'] = ((end_date - df['join_date']).dt.days / 30.44).round(1)

# 4. The cancellation model (Week 10)
cancel_model = smf.logit('cancelled_flag ~ visits_per_month + classes_per_month + distance_km + age + C(membership_type) + C(home_gym)',
                         data=df).fit(disp=0)

# Check all processed correctly
print("Data loaded!")
print(f"{df.shape[0]} members, {df['cancelled_flag'].mean() * 100:.1f}% cancelled")

---
# Block 1: From output to finding
*about 15 minutes*

A notebook output is **not** a finding. A finding in a report has three parts:

| Part | What it does | Example question it answers |
|---|---|---|
| **1. Headline** | States the finding in one sentence | *What did you find?* |
| **2. Evidence** | The key numbers, and the figure that shows them | *How do you know?* |
| **3. So what?** | Why it matters to the business | *Why should I care?* |

In [ ]:
# 👀 Block 1: cancellation rate by membership type
# Watch the demo first, then run this cell yourself

# Cancellation rate (%) and number of members for each membership type
by_type = df.groupby('membership_type').agg(members=('cancelled_flag', 'size'),
                                            cancel_rate=('cancelled_flag', 'mean'))
by_type['cancel_rate'] = (by_type['cancel_rate'] * 100).round(1)
print(by_type.sort_values('cancel_rate', ascending=False))

# A report-ready chart: sorted bars, a clear title, % labels
order = by_type.sort_values('cancel_rate', ascending=False).index
plt.figure(figsize=(9, 4.5))
ax = sns.barplot(x=by_type.loc[order, 'cancel_rate'], y=order, color='steelblue')
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=4)
plt.title('Cancellation Rate by Membership Type')
plt.xlabel('Members who have cancelled (%)')
plt.ylabel('')
plt.xlim(0, 45)
plt.show()

### 👀 Watch: how your tutor wrote it up

> **Figure 1: Cancellation rate by membership type**
>
> **Monthly Flex members are the most likely to cancel.**  
> Over a third (36.5%) of Monthly Flex members have cancelled, against 19.2% of Off-Peak and just 8.1% of Annual members (Figure 1). Monthly Flex is also the largest group, with 1,330 members, so it accounts for most cancellations. Because Monthly Flex members can leave at any time, this is where efforts to keep members will have the most effect.

**Notice:**
- The **headline** is a finding, not a topic ("Monthly Flex members are the most likely to cancel", not "Membership type").
- The **evidence** uses only the key numbers, rounded sensibly, and refers to the figure.
- The **so what** says why it matters to FitLife.
- There is **no code** and **no column names** (i.e. `membership_type`, `cancelled_flag`).

In [ ]:
# 👀 Block 1: cancellation rate by visits per month
# Watch the demo first, then run this cell yourself

# Group members by how often they visit, then the cancellation rate in each group
df['visit_band'] = pd.cut(df['visits_per_month'], bins=[-1, 3, 7, 11, 100], labels=['0-3', '4-7', '8-11', '12+'])

by_visits = df.groupby('visit_band', observed=True).agg(members=('cancelled_flag', 'size'),
                                                        cancel_rate=('cancelled_flag', 'mean'))
by_visits['cancel_rate'] = (by_visits['cancel_rate'] * 100).round(1)
print(by_visits)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=by_visits.index, y=by_visits['cancel_rate'], color='steelblue')
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=4)
plt.title('Cancellation Rate by Visits per Month')
plt.xlabel('Visits per month')
plt.ylabel('Members who have cancelled (%)')
plt.ylim(0, 65)
plt.show()

### ✏️ Draft: write up the visits finding as Figure 2

- **Headline:** what is the finding, in one sentence?
- **Evidence:** which two or three numbers show it best? Refer to Figure 2.
- **So what?** Why does this matter to FitLife? What could they do with it?
- Aim for 60 to 90 words, with no code and no column names.

**Your answer (double-click to edit):**

*Write your answer here.*

### 🔧 Improve: check your draft

Tick each one. If you cannot tick it, edit your draft above.

- [ ] My headline is a **finding** (a full sentence), not a topic
- [ ] I used **two or three** key numbers, rounded sensibly, and referred to **Figure 2**
- [ ] A manager would understand every word: **no code, no column names**
- [ ] I said **why it matters** to FitLife

### 🔴 Go further: a caption that does the work

Some readers only look at the charts. Write a **one-sentence caption** for Figure 2 that a manager could understand without reading the paragraph. Then compare the chart titles in this block with a title like *"Cancellation rate by visit band"*: which helps the reader more, and why?

**Your answer (double-click to edit):**

*Write your answer here.*

---
# Block 2: Model results in plain English
*about 15 minutes*

Your regression output is full of technical terms. Your manager needs to know **what it means**. Two good ways to explain a logistic model:

| In the notebook | In the report |
|---|---|
| Odds ratio **5.54** | Monthly Flex members are **far more likely** to cancel, even after allowing for other factors |
| Odds ratio **0.82** | Each extra visit a month **lowers** the odds of cancelling by about **18%** |
| Predicted probability **0.64** | A member like this has about a **two in three** chance of cancelling |

> 📝 An odds ratio **above 1** means *more* likely, **below 1** means *less* likely. To turn 0.82 into a percentage: 1 - 0.82 = 0.18, so about 18% lower.

In [ ]:
# 👀 Block 2: the cancellation model as a table of odds ratios
# Watch the demo first, then run this cell yourself

# Odds ratios with 95% confidence intervals and p-values (Week 10)
odds = pd.DataFrame({
    'odds_ratio': np.exp(cancel_model.params),
    'lower_95': np.exp(cancel_model.conf_int()[0]),
    'upper_95': np.exp(cancel_model.conf_int()[1]),
    'p_value': cancel_model.pvalues
}).round(3).drop(index='Intercept')
print(odds)

### 👀 Watch: how your tutor wrote it up

> **What makes a member more likely to cancel?** When we look at all the factors together, membership type matters most. Monthly Flex members are far more likely to cancel than Annual members (more than five times the odds), even when they visit as often and live as close to the gym. Members at the Northgate gym are also somewhat more likely to cancel than those at City Centre.

**Notice:**
- "When we look at all the factors together" explains *holding the others constant* without the jargon.
- "More than five times the odds" is used once; the main message is **far more likely**.
- No p-values or confidence intervals in the text. If you mention them, explain them: *"we can be confident this is not due to chance"*.
- Age is **left out**: it was not clearly significant (p = 0.052), and Week 10 showed the U-shape needs age bands.

### ✏️ Draft: explain visits, classes and distance

- Use the odds ratios for **visits_per_month**, **classes_per_month** and **distance_km**.
- For each one, say whether it makes cancelling **more or less** likely, and by roughly how much, in words a manager would understand.
- Aim for 50 to 80 words. No odds ratios in brackets unless you explain them.

**Your answer (double-click to edit):**

*Write your answer here.*

In [ ]:
# 👀 Block 2: predicted chances for two example members
# Watch the demo first, then run this cell yourself

# Two example members: the model predicts the chance that each one cancels
examples = pd.DataFrame({
    'member': ['Member A', 'Member B'],
    'membership_type': ['Monthly Flex', 'Annual'],
    'visits_per_month': [3, 12],
    'classes_per_month': [1, 4],
    'distance_km': [10, 2],
    'age': [30, 30],
    'home_gym': ['City Centre', 'City Centre']
})
examples['chance_of_cancelling'] = (cancel_model.predict(examples) * 100).round(0)
print(examples[['member', 'membership_type', 'visits_per_month', 'distance_km', 'chance_of_cancelling']])

### ✏️ Draft: explain the two example members

- Describe Member A and Member B in words a manager would recognise (not column names).
- Give each one's chance of cancelling as a **percentage** or as **"about 1 in …"**.
- Why might this be more useful to a manager than the odds ratios?

**Your answer (double-click to edit):**

*Write your answer here.*

### 🔧 Improve: check your draft

Tick each one. If you cannot tick it, edit your draft above.

- [ ] No unexplained terms: **odds ratio, p-value, coefficient, significant, logit**
- [ ] Each factor says **more or less likely**, and **roughly how much**
- [ ] No column names (`visits_per_month`): use everyday words (*visits a month*)
- [ ] I explained what *allowing for other factors* means, or avoided needing to

### 🔴 Go further: explain significance in one sentence

Your manager asks: *"What does p < 0.001 mean?"* Write a **one-sentence** answer with no statistical terms. Then write one sentence explaining why the report leaves out the age result.

**Your answer (double-click to edit):**

*Write your answer here.*

---
# Block 3: Recommendations and limitations
*about 15 minutes*

A good recommendation in a data report says **who** to focus on, and backs it up with numbers. It does **not** suggest new initiatives, because your data cannot show whether they would work.

| A good recommendation names… | Example question it answers |
|---|---|
| **The group** | *Which members should FitLife focus on?* |
| **How many** | *How big is the group?* |
| **How likely** | *What is their cancellation rate?* |
| **Share of cancellations** | *How much of the problem do they account for?* |
| **The evidence** | *Which finding shows this? ("Finding 2")* |
| **The limits** | *What can the data **not** tell us?* |

> ❌ **Avoid:** *"FitLife should offer new members a free personal training session."* Nothing in the data shows that personal training would keep members.
>
> ✅ **Instead:** say **which members** FitLife should focus on, and leave the decision about **what to do** to the people who know the business.

In [ ]:
# 👀 Block 3: when do members cancel?
# Watch the demo first, then run this cell yourself

# Members who cancelled: how many months after joining did they leave?
cancelled_members = df[df['cancelled_flag'] == 1]
within_6 = (cancelled_members['tenure_months'] <= 6).mean() * 100
print(f"Median months before cancelling: {cancelled_members['tenure_months'].median():.1f}")
print(f"Cancelled within 6 months of joining: {within_6:.1f}%")

by_month = cancelled_members['tenure_months'].astype(int).value_counts().sort_index()
plt.figure(figsize=(11, 4.5))
plt.bar(by_month.index, by_month.values, color='steelblue')
plt.axvline(6, color='red', linestyle='--')
plt.title('Most cancellations happen in the first six months')
plt.xlabel('Months since joining')
plt.ylabel('Number of cancellations')
plt.show()

In [ ]:
# 👀 Block 3: how big and how risky is each group?
# Watch the demo first, then run this cell yourself

def group_summary(name, in_group):
    """Size, cancellation rate and share of all cancellations for one group of members."""
    return pd.Series({'members': in_group.sum(),
                      '% of all members': round(in_group.mean() * 100, 1),
                      'cancel rate %': round(df.loc[in_group, 'cancelled_flag'].mean() * 100, 1),
                      '% of all cancellations': round(df.loc[in_group, 'cancelled_flag'].sum() / df['cancelled_flag'].sum() * 100, 1)},
                     name=name)

monthly_flex = df['membership_type'] == 'Monthly Flex'
groups = pd.DataFrame([
    group_summary('All members', df['cancelled_flag'] >= 0),
    group_summary('Monthly Flex', monthly_flex),
    group_summary('Monthly Flex, 7 or fewer visits a month', monthly_flex & (df['visits_per_month'] <= 7)),
    group_summary('Any membership, 3 or fewer visits a month', df['visits_per_month'] <= 3),
    group_summary('Northgate gym', df['home_gym'] == 'Northgate'),
    group_summary('Annual, 8 or more visits a month', (df['membership_type'] == 'Annual') & (df['visits_per_month'] >= 8)),
])
groups['members'] = groups['members'].astype(int)
groups

### 👀 Watch: how your tutor wrote a recommendation

> **Recommendation 1: Prioritise Monthly Flex members who visit seven times a month or less.** This group of 607 members is only 20% of FitLife's members, but more than half of them (52.2%) have cancelled, and they account for 45.5% of all cancellations (Findings 1 and 2). Most cancellations happen within the first six months of joining (Finding 3), so these members are most at risk early in their membership. The data shows who is most likely to cancel, not why, so FitLife would need to find out more about these members before deciding how to respond.

**Notice:**
- It names a **group** that FitLife can identify from its own records.
- It gives the **size**, the **cancellation rate** and the **share of all cancellations**, so the manager can see why this group matters.
- It points back to **numbered findings**.
- It does **not** suggest a new initiative, and is **honest** about what the data cannot show.

### ✏️ Draft: a second recommendation

- Choose **one other group** from the table above (not Monthly Flex with 7 or fewer visits).
- Say how many members are in it, their cancellation rate and their share of all cancellations.
- Point back to a **numbered finding**, and say what the data **cannot** tell FitLife.
- Do **not** suggest a new initiative.
- Aim for 50 to 80 words.

**Your answer (double-click to edit):**

*Write your answer here.*

### Limitations

Limitations tell the reader **how far to trust** your findings. Make them **specific** to this analysis and say what could help.

| Type | A FitLife example |
|---|---|
| Association, not cause | Do classes keep members, or do committed members take more classes? |
| Missing information | We do not know **why** members left |
| Small groups | Only 172 members live 10 km or more away |
| Time period | Data ends in January 2025; recent joiners have had less time to cancel |

### ✏️ Draft: one limitation

- Choose **one** limitation from the table (or your own).
- Explain what it means **for FitLife's decisions**, not just in general.
- Say what **further investigation** could help.
- Aim for 40 to 60 words.

**Your answer (double-click to edit):**

*Write your answer here.*

### 🔧 Improve: check your draft

Tick each one. If you cannot tick it, edit your draft above.

- [ ] My recommendation names a **group** that FitLife could identify from its data
- [ ] It gives the group's **size**, **cancellation rate** and **share of all cancellations**
- [ ] It points back to a **numbered finding**
- [ ] It does **not** suggest an initiative that the data cannot support
- [ ] My limitation is **specific to FitLife**, and says what further information could help

### 🔴 Go further: is Northgate a priority?

Northgate has a higher cancellation rate than the other gyms. Run the cell below and look at the Northgate row in the group table. Should Northgate be a priority group? Write a short, **honest** recommendation: what does the data tell you, and what does it **not** tell you?

In [ ]:
# ✍️ Go further: just run this cell
# Cancellation rate (%) at each gym
print((df.groupby('home_gym')['cancelled_flag'].mean() * 100).round(1))

**Your answer (double-click to edit):**

*Write your answer here.*

---
# Block 4: The executive summary
*about 15 minutes*

The executive summary is often the **only** part a busy manager reads. It must make sense **on its own**.

- **Write it last**, once your findings and recommendations are done.
- **150 to 200 words**, in two short paragraphs: what you found, then what to do.
- Include the **two or three most important numbers**.
- **No** method: not "I cleaned the data and ran a regression".

> ❌ **Weak:** *"This report analyses the FitLife dataset. I cleaned the data using pandas, created charts with seaborn and ran a logistic regression model to find factors linked to cancellation."*

In [ ]:
# 👀 Block 4: your key numbers in one place
# Watch the demo first, then run this cell yourself

# The key numbers for the summary, gathered from Blocks 1 to 3
print(f"Members: {df.shape[0]:,}; cancelled: {df['cancelled_flag'].mean() * 100:.1f}%")
print(f"Cancelled within 6 months of joining: {within_6:.0f}%")
print("Cancellation rate by membership type (%):")
print(by_type['cancel_rate'].sort_values(ascending=False).to_string())
print(f"Visit 0-3 times a month: {by_visits.loc['0-3', 'cancel_rate']}%; visit 12+ times: {by_visits.loc['12+', 'cancel_rate']}%")

### ✏️ Draft: the executive summary

- Paragraph 1: **what you found**. The overall picture, then the two or three most important findings, with numbers.
- Paragraph 2: **what FitLife should do**. Your recommendations in one or two sentences, and one honest caveat.
- 150 to 200 words. Use the word counter below.

**Your answer (double-click to edit):**

*Write your answer here.*

### ✍️ Word counter

Paste your executive summary between the three quotation marks, then run the cell.

In [ ]:
# ✍️ Paste your text between the """ marks, then run this cell
my_text = """

"""

print(f"Words: {len(my_text.split())}")

### 🔧 Improve: check your draft

Tick each one. If you cannot tick it, edit your draft above.

- [ ] It makes sense to someone who has **read nothing else**
- [ ] It has the **two or three most important numbers**
- [ ] It says **what to do**, not just what you found
- [ ] It does **not** describe the method (cleaning, code, models)
- [ ] It is **150 to 200 words**

### 🔴 Go further: the 50-word version

Your manager has one minute before a board meeting. Cut your summary to **50 words or fewer** without losing the main message. What did you keep, and why?

**Your answer (double-click to edit):**

*Write your answer here.*

---
## ✅ End of Write-along: Recap

You have practised turning a notebook into a report:

| Block | Skill |
|---|---|
| 1 | A **finding**: headline, evidence, so what, with a numbered figure |
| 2 | **Model results** in plain English: more or less likely, by how much, and predicted chances |
| 3 | **Recommendations about who to prioritise**, with size, rate and share of cancellations, and a **specific limitation** |
| 4 | An **executive summary** that stands alone, written last |

---

### 💾 Before you leave

Make sure your notebook is saved to your Google Drive, if you have not done so already:
- **File → Save a copy in Drive**
- Name it: `Week11_WriteAlong_Notebook_to_Report_YourName.ipynb`

---

### 🔮 Next: the task

After the break you will have time to work on **your own NovaTel notebook**, using the **Assessment 2 Guide**. Have your notebook and your AI log open.